# Assignment 12 — Integrating Azure AI Immersive Reader

Azure AI Immersive Reader helps learners engage with written material by offering adjustable text presentation and read-aloud functionality. A school learning portal can integrate this service to make classroom content, such as a science lesson, more accessible.

The application retrieves `TENANT_ID`, `CLIENT_ID`, `CLIENT_SECRET`, and `SUBDOMAIN` from the current Windows user's environment settings. The `SUBDOMAIN` value identifies the Immersive Reader resource used by the client SDK.

**Sample passage:** "Plants use sunlight to make their own food."

Once the local web page is running, selecting the **Open Immersive Reader** button launches this passage in Immersive Reader. Learners can then use features such as Read Aloud and text spacing.

An active internet connection is required to access the Azure service. In a production deployment, the portal should also authenticate its users before providing an access token.

In [ ]:
import os
import winreg

def user_setting(name):
    with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as key:
        value, _ = winreg.QueryValueEx(key, name)
    return os.path.expandvars(value)

from azure.identity import ClientSecretCredential
from flask import Flask, jsonify, render_template_string

app = Flask(__name__)
credential = ClientSecretCredential(
    tenant_id=user_setting("TENANT_ID"),
    client_id=user_setting("CLIENT_ID"),
    client_secret=user_setting("CLIENT_SECRET"),
)
scope = "https://cognitiveservices.azure.com/.default"
page = """
<!doctype html><meta charset="utf-8">
<title>Reading companion</title>
<h1>Reading companion</h1>
<p id="sample">Plants use sunlight to make their own food.</p>
<button id="read">Open Immersive Reader</button>
<p id="status" role="status"></p>
<script src="https://ircdname.azureedge.net/immersivereadersdk/immersive-reader-sdk.1.4.0.js"></script>
<script>
document.querySelector('#read').onclick = async () => {
  const status = document.querySelector('#status');
  try {
    const response = await fetch('/token', {method: 'POST'});
    if (!response.ok) throw new Error('Azure authentication failed');
    const {token, subdomain} = await response.json();
    const content = document.querySelector('#sample').textContent;
    await ImmersiveReader.launchAsync(token, subdomain, {
      title: 'Photosynthesis',
      chunks: [{content, lang: 'en-US', mimeType: 'text/plain'}]
    });
  } catch (error) {
    status.textContent = error.message;
  }
};
</script>
"""

@app.get("/")
def index():
    return render_template_string(page)

@app.post("/token")
def token():
    access_token = credential.get_token(scope).token
    response = jsonify(
        token=access_token,
        subdomain=user_setting("SUBDOMAIN"),
    )
    response.headers["Cache-Control"] = "no-store"
    return response


Execute the next cell to start the local Flask application. Then visit `http://127.0.0.1:5000` in your browser and select **Open Immersive Reader** to launch the sample passage.

To shut down the local web server, interrupt or stop the running notebook cell.

In [ ]:
app.run(host="127.0.0.1", port=5000)
